# Open-ended survey → theme report (demo)

**Audience:** UX researchers and designers who have exported survey results as a **CSV** and want a **first-pass, frequency-ranked theme table** (keywords + example quotes) before deeper synthesis.

**Method:** TF–IDF + K-means (`survey_theme_report.py`). This **does not replace** careful coding for high-stakes decisions; it **speeds triage** after fieldwork.

See also: `PROBLEM_AND_PUBLISHING.md` in this folder for verification steps and how to share publicly (Colab, Binder, optional web apps).

## 1. Environment (Colab or local)

- **Google Colab:** run the next cell once per session (installs packages in the cloud runtime — nothing to install on your laptop).
- **Local Jupyter:** skip the install cell if you already activated a venv with `requirements.txt`.

In [ ]:
%pip install -q pandas "numpy>=1.24" "scikit-learn>=1.3"

## 2. Get the Python script and sample CSV onto this machine

**You do not need a folder named `W10` on your computer.** `W10` is only where this project lives **inside the GitHub course repo** (`hcde530/W10/`). Colab users never create that by hand—the next cell clones the repo and opens the right subfolder for you.

**Why this step exists:** Your **live link** (e.g. Open in Colab) only opens **this notebook file** in Google’s cloud. It does **not** automatically download the rest of GitHub—so `survey_theme_report.py` and `food_coded.csv` are not present until you put them here.

**What a UX researcher does in practice:**

1. **Easiest (recommended):** Run the next code cell. If the script is missing, it **clones your public GitHub repo once** and moves into the project subfolder (`W10`) where the tool and sample CSV already live.
2. **Alternative:** In Colab’s **Files** sidebar (folder icon), **Upload** `survey_theme_report.py` plus either the sample `food_coded.csv` or their survey CSV—and set paths in step 3 accordingly.
3. **If they use Jupyter on their laptop** and already opened the notebook from inside the cloned project folder, the clone block is skipped.

After this step, “Run the tool” has the same files as if they had opened the project on their computer.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Public repo that contains this notebook under W10/ (edit if you fork).
REPO_URL = "https://github.com/rmchaud/hcde530.git"
CLONE_DIR = Path("/content/hcde530")

here = Path.cwd().resolve()
has_script = (here / "survey_theme_report.py").is_file()

if not has_script and IN_COLAB:
    w10_in_clone = CLONE_DIR / "W10"
    if not (w10_in_clone / "survey_theme_report.py").is_file():
        if CLONE_DIR.exists():
            subprocess.run(["rm", "-rf", str(CLONE_DIR)], check=False)
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
            check=True,
        )
    os.chdir(w10_in_clone)
    w10 = w10_in_clone
elif has_script:
    w10 = here
    os.chdir(w10)
else:
    w10 = here / "W10"
    if (w10 / "survey_theme_report.py").is_file():
        os.chdir(w10)
    else:
        raise FileNotFoundError(
            "Could not find survey_theme_report.py. "
            "Open a terminal, cd into the repo's W10 folder, then launch Jupyter from there—or use Colab."
        )

if str(w10) not in sys.path:
    sys.path.insert(0, str(w10))

print("Working directory:", w10.resolve())

## 3. Run the tool

Default demo uses `food_coded.csv` (Food Choices sample). **Swap** `INPUT_CSV` for any survey export path you have in this runtime (e.g. after uploading a file in Colab).

In [ ]:
from pathlib import Path
from survey_theme_report import main

INPUT_CSV = Path("food_coded.csv")
OUTPUT_CSV = Path("theme_report_notebook.csv")

if not INPUT_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {INPUT_CSV.name}. Add your CSV next to the script or set INPUT_CSV to your uploaded file path."
    )

exit_code = main(
    [
        "--input",
        str(INPUT_CSV),
        "--output",
        str(OUTPUT_CSV),
    ]
)
assert exit_code == 0, exit_code
print("Wrote:", OUTPUT_CSV.resolve())

## 4. Inspect the report

Expected columns: `theme_label`, `frequency`, `percent_of_total`, `representative_quotes`, `keywords`, `rank`.

In [ ]:
import pandas as pd

report = pd.read_csv(OUTPUT_CSV)
display(report.head(15))
print("Rows:", len(report), "| Columns:", list(report.columns))